# 로또 번호 예측

In [2]:
## 로또 정보를 엑셀파일로 저장하고 가져와서 LSTM예측

# 경고 메시지 무시
import warnings
warnings.filterwarnings(action='ignore') 

import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


## 1. 엑셀 파일 로드 (사용자정의함수)
def load_lotto_final(file_path):
    df = pd.read_excel(file_path, skiprows=1)  # 상단 제목 1줄 건너뛰기
    
    try:
        # C열은 인덱스 2 
        win_data = df.iloc[:, [2, 3, 4, 5, 6, 7]].values
        
        # 최신 데이터가 위에 있으므로 과거 순서로 뒤집기
        win_data = np.flip(win_data, axis=0)
        
        # 숫자가 아닌 값(NaN)이 포함된 행은 제거하고 정수로 변환
        win_data = win_data[~np.isnan(win_data).any(axis=1)].astype(int)
        
        # 번호가 1~45 범위를 벗어나는지 체크 (잘못된 열 선택 방지)
        if np.max(win_data) > 45 or np.min(win_data) < 1:
            print("** 경고: 추출된 번호가 1~45 범위를 벗어납니다. 열 위치를 다시 확인해주세요.")
            
        return win_data
    except Exception as e:
        print(f"** 데이터 추출 실패: {e}")
        return None



raw_data = load_lotto_final('../../data/lotto_1_1212.xlsx')

if raw_data is not None:
    print(f"** 데이터 로드 성공: 총 {len(raw_data)}회차 학습 데이터")

    # 2. 원-핫 인코딩 (1~45번 위치 표시)
    encoded_data = np.zeros((len(raw_data), 45))
    for i, row in enumerate(raw_data):
        for num in row:
            if 1 <= num <= 45:
                encoded_data[i, num-1] = 1

    # 3. 시퀀스 생성 (최근 10회차 패턴 학습)
    window_size = 10
    X, y = [], []
    for i in range(len(encoded_data) - window_size):
        X.append(encoded_data[i:i+window_size])
        y.append(encoded_data[i+window_size])
    X, y = np.array(X), np.array(y)

    # 4. LSTM 모델 구성
    model = Sequential([
        LSTM(128, input_shape=(window_size, 45), return_sequences=True),
        Dropout(0.2),
        LSTM(64),
        Dropout(0.2),
        Dense(45, activation='sigmoid') # 각 번호별 당첨 확률 출력
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')

    # 5. 모델 학습 (조기 종료 설정)
    print("** LSTM 모델이 과거 패턴을 학습 중입니다...")
    early_stop = EarlyStopping(monitor='loss', patience=10)
    model.fit(X, y, epochs=100, batch_size=32, callbacks=[early_stop], verbose=0)

    # 6. 다음 회차 번호 예측
    last_seq = encoded_data[-window_size:].reshape(1, window_size, 45)
    prediction = model.predict(last_seq)[0]
    
    # 확률이 가장 높은 상위 6개 번호 추출
    top_6_indices = prediction.argsort()[-6:][::-1]
    predicted_numbers = sorted([idx + 1 for idx in top_6_indices])

    print("\n" + "="*40)
    print(f"** C열 데이터 분석 기반 다음 예측 번호: {predicted_numbers}")
    print("="*40)

** 데이터 로드 성공: 총 1211회차 학습 데이터
** LSTM 모델이 과거 패턴을 학습 중입니다...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 287ms/step

** C열 데이터 분석 기반 다음 예측 번호: [np.int64(3), np.int64(5), np.int64(13), np.int64(14), np.int64(27), np.int64(38)]
